# Práctica: Despliegue y uso de modelos en AI Foundry

Este repositorio contiene la práctica para aprender a desplegar y usar modelos en AI Foundry. La práctica está dividida en 3 partes independientes. Cada parte debe entregarse como un notebook Jupyter (`.ipynb`) con un nombre claro y apropiado.


## 1) Text, JSON y Guardrails
Objetivo: Desplegar un modelo y realizar tres tipos de interacciones:
### 1.1- Generar texto. 
Simple generación de texto con system prompt y user prompt.

⭐ Suma puntos crear un chat interactivo por CLI que persista la memoria a corto plazo.

[Documentación](https://learn.microsoft.com/en-us/azure/foundry/foundry-models/how-to/generate-responses?tabs=python)


In [5]:
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import OpenAI

project_endpoint = "https://pruebauno.services.ai.azure.com/api/projects/projuno"
# Build the base URL: project_endpoint + /openai/v1 (no api-version needed)
base_url = project_endpoint.rstrip("/") + "/openai/v1"

# get_bearer_token_provider returns a callable; call it to get automatic refresh of the token string
credential = DefaultAzureCredential()
token_provider = get_bearer_token_provider(credential, "https://ai.azure.com/.default")
client = OpenAI(
    base_url=base_url,
    api_key=token_provider(),
)   

response = client.responses.create(
    model="gpt-4o-mini", # Replace with your deployment name, not the model ID 
    input="What are the top 3 benefits of cloud computing? Be concise.",
    max_output_tokens=500,
)

print(f"Response: {response.output_text}")
print(f"Status:   {response.status}")
print(f"Output tokens: {response.usage.output_tokens}")

Response: 1. **Scalability**: Easily adjust resources and storage capacity based on demand without significant upfront investment.

2. **Cost Efficiency**: Pay-as-you-go pricing models reduce capital expenses, allowing businesses to optimize costs.

3. **Accessibility**: Access data and applications from anywhere with an internet connection, enhancing collaboration and flexibility.
Status:   completed
Output tokens: 68


### 1.2- Generar respuesta estructurada en formato JSON.
Generación de respuesta estructurada en JSON.

[Documentación](https://learn.microsoft.com/en-us/azure/foundry/openai/how-to/structured-outputs?tabs=python-secure%2Cdotnet-entra-id&pivots=programming-language-python)


In [6]:
from pydantic import BaseModel
from openai import OpenAI
from azure.identity import DefaultAzureCredential, get_bearer_token_provider

token_provider = get_bearer_token_provider(
    DefaultAzureCredential(), "https://ai.azure.com/.default"
)

client = OpenAI(  
  base_url = "https://pruebauno.services.ai.azure.com/api/projects/projuno/openai/v1",  
  api_key=token_provider(),
)

class CalendarEvent(BaseModel):
    name: str
    date: str
    participants: list[str]
    equipo_Ganador: str

completion = client.beta.chat.completions.parse(
    model="gpt-4o-mini", # replace with the model deployment name of your gpt-4o 2024-08-06 deployment
    messages=[
        {"role": "system", "content": "Extract the event information."},
        {"role": "user", "content": "El Atleti va a ganar la Champions League y Mario que es del barça va a llorar."},
    ],
    response_format=CalendarEvent,
)

event = completion.choices[0].message.parsed

print(event)
print(completion.model_dump_json(indent=2))

name='Champions League Final' date='2023-06-10' participants=['Atleti', 'Barça', 'Mario'] equipo_Ganador='Atleti'
{
  "id": "chatcmpl-DTlNNYvVFwB5DfdhcZBqKeOcmmKmx",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "{\"name\":\"Champions League Final\",\"date\":\"2023-06-10\",\"participants\":[\"Atleti\",\"Barça\",\"Mario\"],\"equipo_Ganador\":\"Atleti\"}",
        "refusal": null,
        "role": "assistant",
        "annotations": [],
        "audio": null,
        "function_call": null,
        "tool_calls": null,
        "parsed": {
          "name": "Champions League Final",
          "date": "2023-06-10",
          "participants": [
            "Atleti",
            "Barça",
            "Mario"
          ],
          "equipo_Ganador": "Atleti"
        }
      },
      "content_filter_results": {
        "hate": {
          "filtered": false,
          "severity": "safe"
        },
        "protecte

c:\Users\akook\Desktop\foundry-ejers\practicaFoundry\.venv\Lib\site-packages\pydantic\main.py:528: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=CalendarEvent(name='Champ...equipo_Ganador='Atleti'), input_type=CalendarEvent])
  return self.__pydantic_serializer__.to_json(


### 1.3- Implementar y demostrar Guardrails. 
Crear Guardrails para el modelo, documentar el proceso y hacer pruebas contra el modelo.

[Documentación](https://learn.microsoft.com/en-us/azure/foundry/guardrails/how-to-create-guardrails?tabs=python)

### Entregable: 
Notebook con código que muestre la llamada al endpoint del modelo para cada caso, ejemplos de prompts, validación del JSON recibido y una sección que muestre cómo se configuran y activan los Guardrails.

In [ ]:
# Guardrails testing with Azure credentials

from openai import OpenAI
from azure.identity import DefaultAzureCredential, get_bearer_token_provider

# Set up credentials using DefaultAzureCredential
token_provider = get_bearer_token_provider(
    DefaultAzureCredential(), "https://ai.azure.com/.default"
)

client = OpenAI(  
  base_url = "https://pruebauno.services.ai.azure.com/api/projects/projuno/openai/v1",  
  api_key=token_provider(),
)

# Test prompts for guardrails severity detection
test_prompts = [
    "What is the capital of France?",
    "Explain how artificial intelligence works",
    "Tell me about cloud computing benefits"
]

print("Testing Guardrails with different prompts:\n")
for prompt in test_prompts:
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "You are a helpful assistant."},
                {"role": "user", "content": prompt}
            ],
            max_tokens=200
        )
        print(f"Prompt: {prompt}")
        print(f"Response: {response.choices[0].message.content}")
        print(f"Severity: Low (content not filtered)\n")
    except Exception as e:
        print(f"Prompt: {prompt}")
        print(f"Error/Severity: {str(e)}\n")

ValueError: Please set AZURE_OPENAI_API_KEY environment variable

---

## 2) Reasoning y Function Calling
Objetivo: Practicar con modelos razonadores y ver el function calling

### 2.1- Razonamiento
Desplegar un modelo razonador y parametrizar distintos grados de razonamiento (low, medium, high)

[Documentación](https://learn.microsoft.com/en-us/azure/foundry/openai/how-to/reasoning?tabs=csharp%2Cgpt-5)

In [ ]:
from openai import OpenAI
from azure.identity import DefaultAzureCredential, get_bearer_token_provider

token_provider = get_bearer_token_provider(
    DefaultAzureCredential(), "https://ai.azure.com/.default"
)

client = OpenAI(  
  base_url = "https://pruebauno.services.ai.azure.com/api/projects/projuno/openai/v1",  
  api_key=token_provider,
)

response = client.chat.completions.create(
  model="gpt-4o-mini", # replace with your model deployment name
    messages=[
        {"role": "user", "content": "What steps should I think about when writing my first Python API?"},
    ],
    max_completion_tokens = 5000

)

print(response.model_dump_json(indent=2))

{
  "id": "chatcmpl-DTRMEhwvdpHQBGReYngqRTEQeqNQp",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "Creating your first Python API can be an exciting journey! Here's a step-by-step guide to help you through the process:\n\n### 1. Define Your API's Purpose\n- **Identify the Problem**: Clearly define what problem your API will solve or what functionality it will provide.\n- **Target Audience**: Understand who will use your API and what they need from it.\n\n### 2. Choose the Right Tools and Frameworks\n- **Framework Selection**: Consider popular Python frameworks for API development, such as:\n  - **Flask**: Lightweight and easy to get started with.\n  - **Django REST Framework**: More feature-rich, suitable for larger projects.\n  - **FastAPI**: Asynchronous and very fast, good for modern applications.\n  \n### 3. Design the API\n- **Endpoints**: Plan the endpoints (URI paths) your API will expose.\n- *

In [ ]:
import os
from openai import OpenAI

# Load API Key from environment variable
api_key = os.getenv("AZURE_OPENAI_API_KEY")
base_url = "https://pruebauno.services.ai.azure.com/api/projects/projuno/openai/v1"

if not api_key:
    raise ValueError("Please set AZURE_OPENAI_API_KEY environment variable")

client = OpenAI(
    api_key=api_key,
    base_url=base_url,
)

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": "What steps should I think about when writing my first Python API?"},
    ],
    max_tokens=5000
)

print(response.model_dump_json(indent=2))

{
  "id": "chatcmpl-DTRO5dPr7lFCAfYfBdiTVkkJqIoee",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "Creating your first Python API can be an exciting project! Below are the essential steps you should consider when writing your API:\n\n### 1. Define the Purpose of Your API\n   - Determine what your API will do. What kind of data will it manage? What endpoints will it offer? Understanding the core functionality is crucial.\n\n### 2. Choose a Framework\n   - Decide which web framework you want to use. Popular choices for building APIs in Python include:\n     - **Flask**: Lightweight and easy to use for small to medium-sized applications.\n     - **Django** + Django REST Framework: Feature-rich and suited for larger applications, comes with a lot of built-in functionalities.\n     - **FastAPI**: Great for asynchronous programming and automatic generation of OpenAPI documentation.\n\n### 3. Set Up Your Env

In [ ]:
import os
from openai import OpenAI

# Load API Key from environment variable
api_key = os.getenv("AZURE_OPENAI_API_KEY")
base_url = "https://pruebauno.services.ai.azure.com/api/projects/projuno/openai/v1"

if not api_key:
    raise ValueError("Please set AZURE_OPENAI_API_KEY environment variable")

client = OpenAI(
    base_url=base_url,
    api_key=api_key
)

# Test with different levels of reasoning (if supported by the model)
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a helpful assistant that provides detailed explanations."},
        {"role": "user", "content": "What steps should I think about when writing my first Python API?"},
    ],
    max_tokens=1000,
)

print(response.model_dump_json(indent=2))

{
  "id": "chatcmpl-DTRPxvEmWAReFFAYZdjd4vCobmclx",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "Creating your first Python API is an exciting project! Below are the steps you should consider, along with some best practices to help you along the way:\n\n### 1. Define Your API Requirements\n\n- **Identify Use Case:** Understand what problem your API will solve. Define the core functionalities and how clients will interact with it.\n- **Data Entities:** Determine the data structure you will use (e.g., JSON format). Identify the entities, their attributes, and relationships.\n\n### 2. Choose Frameworks and Libraries\n\n- **Framework Selection:** Decide on a web framework. Popular choices for building APIs in Python include:\n    - **Flask:** Simple and lightweight; great for small to medium APIs.\n    - **Django Rest Framework (DRF):** Part of Django; best for larger applications and comes with many bu

In [ ]:
import os
from openai import OpenAI

# Load API Key from environment variable
api_key = os.getenv("AZURE_OPENAI_API_KEY")
base_url = "https://pruebauno.services.ai.azure.com/api/projects/projuno/openai/v1"

if not api_key:
    raise ValueError("Please set AZURE_OPENAI_API_KEY environment variable")

client = OpenAI(
    api_key=api_key,
    base_url=base_url,
)

# Test with different levels of reasoning (if supported by the model)
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a helpful assistant that provides detailed explanations about Python APIs."},
        {"role": "user", "content": "What steps should I think about when writing my first Python API?"},
    ],
    max_tokens=2000,
)

print(response.model_dump_json(indent=2))

{
  "id": "chatcmpl-DTRRcJ5iXlX4lmap8KmCSqUKHycio",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "Creating your first Python API can be a rewarding experience. It generally involves various steps from planning your API to deploying it for others to use. Here’s a detailed guide to help you through the process:\n\n### 1. **Planning Your API**\n\n   - **Define the Purpose**: Determine what the API will do. Will it allow users to retrieve data, create resources, or perform actions?\n   - **Identify Resources**: List the main entities (resources) of your API. For example, if it's a blog API, resources might include posts, comments, and users.\n   - **Design Endpoints**: Draft out the endpoints and their respective methods (GET, POST, PUT, DELETE). For example:\n     - `GET /posts` - Retrieve a list of posts.\n     - `POST /posts` - Create a new post.\n     - `GET /posts/{id}` - Retrieve a specific post by

### 2.2- Function calling
Activar un motor de búsqueda web para probar llamadas a funciones (`function calling`) que recuperen información externa.

[Documentación](https://learn.microsoft.com/en-us/azure/foundry/openai/how-to/web-search)

⭐ Suma puntos usar deep research o hacer function calling con una función custom.


### Entregable: 
Notebook que muestre:
	- Ejemplos comparativos (misma tarea con distintos niveles de razonamiento).
	- Integración del web search y ejemplo de `function calling` que combine resultados externos con la respuesta del modelo.

In [ ]:
from openai import OpenAI

# Use the client already configured in previous cells
response = client.responses.create(   
  model="gpt-4o-mini", # Replace with your model deployment name
  tools=[{"type": "web_search"}], 
  input="Please perform a web search on the latest trends in renewable energy"
)

print(response.output_text)

OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable

In [ ]:
from openai import OpenAI
from azure.identity import DefaultAzureCredential, get_bearer_token_provider

token_provider = get_bearer_token_provider(
    DefaultAzureCredential(), "https://ai.azure.com/.default"
)

client = OpenAI(  
  base_url = "https://pruebauno.services.ai.azure.com/api/projects/projuno/openai/v1",  
  api_key=token_provider,
)

response = client.responses.create(   
  model="gpt-4o-mini", # Replace with your model deployment name
  tools=[{"type": "web_search"}], 
  input="Please perform a web search on the latest trends in renewable energy"
)

print(response.output_text)

---

## 3) Modelos Multimodales
### Objetivo: 
Desplegar un modelo multimodal y probar interacciones que involucren imágenes, audio y/o texto combinado (por ejemplo: describir una imagen, transcribir audio y responder preguntas sobre su contenido, etc.).

[Documentación](https://learn.microsoft.com/en-us/azure/foundry-classic/foundry-models/how-to/use-chat-multi-modal?context=%2Fazure%2Ffoundry%2Fcontext%2Fcontext&pivots=programming-language-python)

### Entregable: 
Notebook con llamadas al endpoint multimodal mostrando varios ejemplos: subida/consulta de imágenes, audios y prompts mixtos; incluir control de formatos y manejo de respuestas (texto y/o estructuras).

---

Formato y criterios de entrega
- Cada parte debe entregarse como un notebook `.ipynb` autocontenido que incluya:
	- Sección de configuración / credenciales (explicando cómo configurar variables de entorno localmente).
	- Código reproducible conecta a un modelo ya desplegado, realiza las llamadas y procesa las respuestas.
	- Celdas de explicación y resultados visibles (salidas, figuras, JSON validados).
	- Una sección final de conclusiones y problemas encontrados.